# 12 · H4′: глобальный срез против условного

**Зачем** (решение Павла 2026-10-31, `docs/HYPOTHESES.md` H4′). H4 утверждает: вмешательство в активации выигрывает
у дообучения на **условных** изменениях и проигрывает на **глобальных**. Для классического steering на условном срезе
это уже не так (LoRA сильно лучше CAA). Проверяем ту же идею для методов проекта — аффинных поправок в точке вызова
без градиента (`linmap`, `linmap_prompt`) — против LoRA.

**Глобальный срез** (генератор `g1`, `docs/METRICS.md` «Глобальный срез»). Тот же магазин и те же ответы `check_stock`
в двух ветках (подходит / не подходит), но правильное действие одно: сначала спросить пользователя (`ask_user`).
Правило в промпте не написано — его учат из данных. Контрольные пары: пользователь уже ответил, верное действие
зависит от ответа, спросить снова — ошибка. Принятие правила = столбец `sensitivity`, сохранность = `specificity`.

Учитель `linmap_prompt` здесь — правило словами в system (`policy_prompt`). CAA (`steering`) — справочно.
Условный срез — прогон ноутбука 11 (тот же набор методов, то же правило отбора). Перенос — свежий seed 606.
Результат **предварительный** (test не открыт).


In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
# Правило отбора — то же, что для условного среза (запись в журнале до этого прогона).
SELECT = "val+loto"
cfg = ExperimentConfig.make(
    name="global-slice-qwen3-1.7b", hypothesis="explore", notebook="notebooks/12_global_slice.ipynb",
    notes="H4′: глобальный срез g1 (правило ask_user) — linmap, linmap_prompt (учитель policy_prompt), LoRA; CAA справочно.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "g1",
        "data.stratify_pair_type": True,
        "eval.methods": ["baseline", "policy_prompt", "steering", "lora", "linmap", "linmap_prompt"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "act.select": SELECT,
        "act.teacher": "policy_prompt",
        "act.ranks": [8],
        "act.alphas": [1.0, 1.5, 2.0, 3.0, 4.0],
        "episode.enabled": False,
        "transfer.seed": 606,
        "transfer.episode_methods": [],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")


### Подбор на глобальном срезе

In [ ]:
f = exp.state["fitted"]
lp = f["linmap_prompt"]
print(f"учитель {lp.get('teacher')}: успех на train {lp.get('teacher_train_success', float('nan')):.3f} "
      f"(база {lp.get('base_train_success', float('nan')):.3f})")
cols = ["layer", "rank", "alpha", "success", "loto_success", "score", "sensitivity", "specificity", "spec_drop"]
for m in ("linmap", "linmap_prompt"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    r, rv = f[m]["row"], f[m].get("row_val") or f[m]["row"]
    display(Markdown(f"**{m}** — выбрано L{r['layer']}/α{r['alpha']}; по одному val: L{rv['layer']}/α{rv['alpha']}"))
    display(t.sort_values("score" if "score" in t else "success", ascending=False)[[c for c in cols if c in t]].head(6).round(3))


### Перенос (seed 606): знакомые и новые типы требований

In [ ]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))


### H4′: условный срез (ноутбук 11) против глобального

Δ = успех метода − успех LoRA, парно по задачам; I = Δ_усл − Δ_глоб, интервал уровня по Холму (97.5% на два метода).
Знакомые типы — предварительный вывод по H4′; новые типы — разведка.

In [ ]:
from steering_tool_feedback import crossslice

COND_RUN = exp.store.results / "runs" / "20261031-160942_loto-select-qwen3-1.7b_bd79ce"   # ноутбук 11
METHODS = {"linmap": ("linmap", "linmap"), "linmap_prompt": ("linmap_prompt", "linmap_prompt")}
h4p = {}
for g in ("seen", "unseen"):
    out = crossslice.compare(COND_RUN, exp.dir, f"transfer:{g}", methods=METHODS,
                             tol=cfg.steer.max_specificity_drop, n_boot=cfg.eval.n_boot)
    h4p[g] = out
    display(Markdown(f"**{g}** (уровень {out['level']:.3f}) — H4′ {out['verdict'].label}: {out['verdict'].reason}"))
    display(pd.DataFrame(out["rows"]).drop(columns=["cond_name", "glob_name"]).round(3))
exp._save_json("h4_prime.json", {g: {**{k: v for k, v in o.items() if k != "verdict"},
                                     "verdict": o["verdict"].to_dict()} for g, o in h4p.items()})


## Вывод → журнал

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))
